Checklist

 - Add GNN layer (currently 2, need 3)
 - Reduce dimension of GNN layer
 - Add relevant features in featurizer
 - Check dimensions at each layer
 - Hyperparameter tuning

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from rdkit import Chem
from rdkit import RDLogger
from rdkit.Chem.Draw import IPythonConsole
from rdkit.Chem.Draw import MolsToGridImage

warnings.filterwarnings("ignore")
RDLogger.DisableLog("rdApp.*")

np.random.seed(42)

In [1]:
import torch
from torch.nn import Linear
import torch.nn.functional as F 
from torch_geometric.nn import GCNConv, TopKPooling
from torch_geometric.nn import global_mean_pool as gap, global_max_pool as gmp
embedding_size = 64

KeyboardInterrupt: 

In [ ]:
class Featurizer:
    def __init__(self, allowable_sets):
        self.dim = 0
        self.features_mapping = {}
        for k, s in allowable_sets.items():
            s = sorted(list(s))
            self.features_mapping[k] = dict(zip(s, range(self.dim, len(s) + self.dim)))
            self.dim += len(s)

    def encode(self, inputs):
        output = np.zeros((self.dim,))
        for name_feature, feature_mapping in self.features_mapping.items():
            feature = getattr(self, name_feature)(inputs)
            if feature not in feature_mapping:
                continue
            output[feature_mapping[feature]] = 1.0
        return output


class AtomFeaturizer(Featurizer):
    def __init__(self, allowable_sets):
        super().__init__(allowable_sets)

    def symbol(self, atom):
        return atom.GetSymbol()

    def n_valence(self, atom):
        return atom.GetTotalValence()

    def n_hydrogens(self, atom):
        return atom.GetTotalNumHs()

    def hybridization(self, atom):
        return atom.GetHybridization().name.lower()


class BondFeaturizer(Featurizer):
    def __init__(self, allowable_sets):
        super().__init__(allowable_sets)
        self.dim += 1

    def encode(self, bond):
        output = np.zeros((self.dim,))
        if bond is None:
            output[-1] = 1.0
            return output
        output = super().encode(bond)
        return output

    def bond_type(self, bond):
        return bond.GetBondType().name.lower()

    def conjugated(self, bond):
        return bond.GetIsConjugated()


atom_featurizer = AtomFeaturizer(
    allowable_sets={
        "symbol": {"B", "Br", "C", "Ca", "Cl", "F", "H", "I", "N", "Na", "O", "P", "S"},
        "n_valence": {0, 1, 2, 3, 4, 5, 6},
        "n_hydrogens": {0, 1, 2, 3, 4},
        "hybridization": {"s", "sp", "sp2", "sp3"},
    }
)

#29 dimensions

bond_featurizer = BondFeaturizer(
    allowable_sets={
        "bond_type": {"single", "double", "triple", "aromatic"},
        "conjugated": {True, False},
    }
)

#6 dimensions

node_dim = 29
edge_dim = 6

In [ ]:
import torch
import numpy as np
from rdkit import Chem
from torch_geometric.data import Data

# Ensure you define atom_featurizer and bond_featurizer before using them

def molecule_from_smiles(smiles):
    """Convert SMILES to RDKit molecule with error handling."""
    molecule = Chem.MolFromSmiles(smiles, sanitize=False)
    flag = Chem.SanitizeMol(molecule, catchErrors=True)
    if flag != Chem.SanitizeFlags.SANITIZE_NONE:
        Chem.SanitizeMol(molecule, sanitizeOps=Chem.SanitizeFlags.SANITIZE_ALL ^ flag)
    Chem.AssignStereochemistry(molecule, cleanIt=True, force=True)
    return molecule

def graph_from_molecule(molecule):
    """Convert RDKit molecule to PyTorch Geometric graph representation."""
    atom_features = []
    bond_features = []
    edge_index = []
    edge_attr = []

    for atom in molecule.GetAtoms():
        atom_features.append(atom_featurizer.encode(atom))  # Encode atom features

    for bond in molecule.GetBonds():
        start, end = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edge_index.append([start, end])
        edge_index.append([end, start])  # Ensure undirected edges
        bond_features.append(bond_featurizer.encode(bond))  # Bond features
        bond_features.append(bond_featurizer.encode(bond))  # Reverse edge

    # Convert to PyTorch tensors
    x = torch.tensor(atom_features, dtype=torch.float)
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(bond_features, dtype=torch.float)

    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

def graphs_from_smiles(smiles_list):
    """Convert a list of SMILES strings to PyTorch Geometric Data objects."""
    graphs = []
    for smiles in smiles_list:
        molecule = molecule_from_smiles(smiles)
        graph = graph_from_molecule(molecule)
        graphs.append(graph)
    return graphs  # This can be used with a PyG DataLoader


In [ ]:
from pymatgen.core import Structure
from pymatgen.io.cif import CifParser
from pymatgen.analysis.local_env import CutOffDictNN
from scipy.spatial import distance_matrix
from rdkit import Chem
from rdkit.Chem import AllChem, Draw

import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

def cif_to_mol(cif_file):
    structure = Structure.from_file(cif_file)

    mol = Chem.RWMol()  # Create an editable molecule in RDKit

    for site in structure:
        element = site.specie.symbol  # Get atomic symbol (e.g., "C", "O")
        atom = Chem.Atom(element)  # Create an RDKit atom
        mol.AddAtom(atom)  # Add it to the molecule

    #print(structure[0].coords)

    cutoff = 1.8  # Example: typical bond length for C-C or C-H

    for i in range(len(structure)):
        for j in range(i + 1, len(structure)):
            dist = structure.get_distance(i, j)
            if dist < cutoff:  # Only consider distances within the cutoff
                #print(f"Bond between atom {i} and atom {j}: {dist:.3f} Å")
                if mol.GetBondBetweenAtoms(i, j) is None:  
                    mol.AddBond(i, j, Chem.BondType.SINGLE)  # Only add if not present

    conf = Chem.Conformer(mol.GetNumAtoms())  # Create a conformer

    for idx, site in enumerate(structure):
        coord = site.coords  # Cartesian coordinates (x, y, z)
        conf.SetAtomPosition(idx, coord)  # Assign position

    mol.AddConformer(conf)  # Add conformer to molecule

    return mol

In [ ]:
from torch_geometric.data import Data

class PairedData(Data):
    def __init__(self, data1, data2, y):
        super().__init__()
        self.x1 = data1.x
        self.edge_index1 = data1.edge_index
        self.edge_attr1 = data1.edge_attr
        
        self.x2 = data2.x
        self.edge_index2 = data2.edge_index
        self.edge_attr2 = data2.edge_attr
        
        self.y = y  # Target value for the pair

    def __inc__(self, key, value, *args, **kwargs):
        """Ensures proper indexing when batching."""
        if key == "edge_index1":
            return self.x1.shape[0] if self.x1 is not None else 0
        if key == "edge_index2":
            return self.x2.shape[0] if self.x2 is not None else 0
        return super().__inc__(key, value, *args, **kwargs)

In [ ]:
import pandas as pd
import os

df = pd.read_csv() #TODO

file_path_cif = r"/" #TODO
file_path_sdf = r"/" #TODO

materials = []
drugs = []

for cif in df["cif"]:
    file = os.path.join(file_path_cif, cif + ".cif")
    print(f"Processing CIF: {file}")  # Progress tracking
    mol = cif_to_mol(file)
    materials.append(graph_from_molecule(mol))
print("Finished processing all CIF files.\n")

for sdf in df["sdf"]:
    file = os.path.join(file_path_sdf, sdf + ".sdf")
    print(f"Processing SDF: {file}")  # Progress tracking
    supplier = Chem.SDMolSupplier(file)
    mol = supplier[0]
    if mol is None:
        print(f"Warning: Failed to read molecule from {file}")  # Handle errors
    drugs.append(graph_from_molecule(mol))
print("Finished processing all SDF files.\n")

print("Pairing materials and drugs...")
paired_data_list = []
for i, (mat, drug, target) in enumerate(zip(materials, drugs, df["y"])):
    print(f"Pairing {i+1}/{len(df)}: Material-{i}, Drug-{i}, Target-{target}")
    paired_data_list.append(PairedData(mat, drug, target))
print("Finished pairing all data.")


In [ ]:
from sklearn.model_selection import train_test_split

# Define train-test split ratio (e.g., 80% train, 20% test)
train_dataset, test_dataset = train_test_split(paired_data_list, test_size=0.2, random_state=42)

print(f"Training set size: {len(train_data)}")
print(f"Testing set size: {len(test_data)}")

In [ ]:
from torch_geometric.loader import DataLoader

dataloader = DataLoader(paired_data_list, batch_size=32, shuffle=True)
for batch in dataloader:
    print(batch)  # Check if batching works correctly

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.nn import global_mean_pool as gap, global_max_pool as gmp

class MPNN(MessagePassing):
    def __init__(self, in_dim, edge_dim, out_dim, hidden_dim=32, aggr="mean"):
        super().__init__(aggr=aggr)  # "mean", "sum", or "max" aggregation

        # MLP for message transformation
        self.mlp = nn.Sequential(
            nn.Linear(in_dim + edge_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim)
        )

    def forward(self, x, edge_index, edge_attr):
        # x: Node features (num_nodes, in_dim)
        # edge_index: Graph connectivity (2, num_edges)
        # edge_attr: Edge features (num_edges, edge_dim)

        return self.propagate(edge_index, x=x, edge_attr=edge_attr)

    def message(self, x_j, edge_attr):
        # x_j: Neighbor node features
        # edge_attr: Edge features

        # Concatenate node and edge features
        msg_input = torch.cat([x_j, edge_attr], dim=1)

        # Transform message using MLP
        return self.mlp(msg_input)

    def update(self, aggr_out):
        # Update node representations after aggregation
        return aggr_out


class CustomGNN(torch.nn.Module):
    def __init__(self, node_in_dim, edge_dim, hidden_dim, out_dim):
        super().__init__()
        self.conv1 = MPNN(node_in_dim, edge_dim, hidden_dim)
        self.conv2 = MPNN(hidden_dim, edge_dim, hidden_dim)
        self.conv3 = MPNN(hidden_dim, edge_dim, out_dim)

    def forward(self, x, edge_index, edge_attr):
        x = self.conv1(x, edge_index, edge_attr)
        x = F.relu(x)
        x = self.conv2(x, edge_index, edge_attr)
        x = F.rely(x)
        x = self.conv3(x, edge_index, edge_attr) 

        # Compute mean and max pooling
        hidden = torch.cat([gmp(x), gap(x)], dim=1)

        return hidden


In [ ]:
class MLPMessagePassing(torch.nn.Module):
    def __init__(self, model, in_dim):
        super().__init__()
        self.GNN = model  # Pass an existing GNN model

        self.MLP = nn.Sequential(
            nn.Linear(in_dim, in_dim // 2),  # First linear layer
            nn.ReLU(),  # Activation function
            nn.Linear(in_dim // 2, 1)  # Output layer
        )

    def forward(self, batch): #Equivariance 
        # Apply GNN to both sets of graph inputs
        out1 = self.GNN(batch.x, batch.edge_index, batch.edge_attr)
        out2 = self.GNN(batch.x2, batch.edge_index2, batch.edge_attr2)

        # Concatenate graph representations
        combined_out = torch.cat([out1, out2], dim=1)

        # Apply MLP for final output
        result = self.MLP(combined_out)

        return result

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.loader import DataLoader  # Assuming PyG DataLoader

# Define training function
def train(model, train_loader, optimizer, criterion, device):
    model.train()  # Set model to training mode
    total_loss = 0

    for batch in train_loader:
        batch = batch.to(device)  # Move batch to GPU if available
        
        optimizer.zero_grad()  # Reset gradients
        output = model(batch)  # Forward pass

        loss = torch.sqrt(criterion(output, batch.y))  # RMSE Loss
        loss.backward()  # Backpropagation
        optimizer.step()  # Update weights

        total_loss += loss.item()
    
    return total_loss / len(train_loader)  # Return average loss

# Define evaluation function
def evaluate(model, val_loader, criterion, device):
    model.eval()  # Set model to evaluation mode
    total_loss = 0

    with torch.no_grad():  # Disable gradient tracking
        for batch in val_loader:
            batch = batch.to(device)
            output = model(batch)

            loss = torch.mse_loss(output, batch.y, reduction='mean').sqrt() # RMSE Loss
            total_loss += loss.item()
    
    return total_loss / len(val_loader)

# Model, Optimizer, and Loss Function
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

gnn_model = CustomGNN(node_dim, edge_dim, hidden_dim=256, out_dim=128).to(device)
model = MLPMessagePassing(gnn_model, in_dim=256).to(device) #in_dim = out_dim x 2

optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
criterion = nn.MSELoss()  # RMSE = sqrt(MSE)

# Training Loop
num_epochs = 50
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
val_loader = DataLoader(test_dataset, batch_size=2, shuffle=False)

for epoch in range(1, num_epochs + 1):
    train_loss = train(model, train_loader, optimizer, criterion, device)
    val_loss = evaluate(model, val_loader, criterion, device)

    print(f"Epoch {epoch}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")

print("Training Complete! ✅")
